# Day 2: Validation Rules and Data Quality Audit

This notebook shows how to define validation rules and create a simple audit report.

> Focus on detecting/reporting issues, not full cleaning implementation.


In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/raw/week2_hr_employee_records_messy.csv", "data/raw/week2_hr_employee_records_messy.csv"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Downloaded: data\raw\week2_hr_employee_records_messy.csv
Bootstrap complete.


## Step 1 - Define validation rules


In [4]:
import pandas as pd

df = pd.read_csv('data/raw/week2_hr_employee_records_messy.csv')

valid_departments={'D10','D20','D30','D40'}
valid_contract={'full-time','part-time','contractor','intern'}
valid_mgr={'M001','M002','M003','M004'}
rules={
'required_employee_id': df['employee_id'].isna() | (df['employee_id'].astype(str).str.strip()==''),
'age_between_0_120': (pd.to_numeric(df['age'], errors='coerce')<0) | (pd.to_numeric(df['age'], errors='coerce')>120) | pd.to_numeric(df['age'], errors='coerce').isna(),
'salary_non_negative': (pd.to_numeric(df['salary'], errors='coerce')<0) | pd.to_numeric(df['salary'], errors='coerce').isna(),
'hire_date_valid': pd.to_datetime(df['hire_date'], errors='coerce', format='mixed').isna(),
'duplicate_employee_id': df.duplicated(subset=['employee_id'], keep=False),
'known_department': ~df['department_id'].isin(valid_departments),
'contract_standardized': ~df['contract_type'].str.lower().isin(valid_contract),
'manager_reference_exists': df['manager_id'].isna() | ~df['manager_id'].isin(valid_mgr),
}
pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T

,affected_rows
required_employee_id,1
age_between_0_120,1
salary_non_negative,2
hire_date_valid,1
duplicate_employee_id,2
known_department,2
contract_standardized,1
manager_reference_exists,2


## Step 2 - Row-level evidence


In [5]:
for r,m in rules.items():
    print('\n', '='*70)
    print(r, 'affected rows:', int(m.sum()))
    display(df.loc[m].head(5))



required_employee_id affected rows: 1


,employee_id,department_id,hire_date,birth_date,age,salary,contract_type,email,manager_id,last_updated
7,NaN,D20,2025-01-10,2001-01-10,24,41000.0,part-time,gina@corp.com,M003,2026-03-05



age_between_0_120 affected rows: 1


,employee_id,department_id,hire_date,birth_date,age,salary,contract_type,email,manager_id,last_updated
9,E009,D40,2023-10-10,1960-10-10,150,58000.0,full-time,ian@corp.com,NaN,2026-03-08



salary_non_negative affected rows: 2


...

,employee_id,department_id,hire_date,birth_date,age,salary,contract_type,email,manager_id,last_updated
2,E003,NaN,2021-01-10,2005-01-01,20,38000.0,part-time,carla@corp.com,M002,2026-03-02
4,E005,D99,2023-02-28,1984-06-06,41,71000.0,Full Time,eva@corp.com,M999,2026-03-04



contract_standardized affected rows: 1


,employee_id,department_id,hire_date,birth_date,age,salary,contract_type,email,manager_id,last_updated
4,E005,D99,2023-02-28,1984-06-06,41,71000.0,Full Time,eva@corp.com,M999,2026-03-04



manager_reference_exists affected rows: 2


,employee_id,department_id,hire_date,birth_date,age,salary,contract_type,email,manager_id,last_updated
4,E005,D99,2023-02-28,1984-06-06,41,71000.0,Full Time,eva@corp.com,M999,2026-03-04
9,E009,D40,2023-10-10,1960-10-10,150,58000.0,full-time,ian@corp.com,NaN,2026-03-08


## Step 3 - Build audit report


In [6]:
sev={
'required_employee_id':'High','age_between_0_120':'High','salary_non_negative':'High',
'hire_date_valid':'Medium','duplicate_employee_id':'High','known_department':'Medium',
'contract_standardized':'Low','manager_reference_exists':'Medium'}
rec={
'required_employee_id':'Escalate and resolve missing IDs before joins.',
'age_between_0_120':'Verify with HR master records and correct source data.',
'salary_non_negative':'Investigate extract mapping and payroll source rules.',
'hire_date_valid':'Enforce standard date format at entry point.',
'duplicate_employee_id':'Investigate key-generation process and dedup policy for next chapter.',
'known_department':'Reconcile against department reference table.',
'contract_standardized':'Align labels to canonical categories.',
'manager_reference_exists':'Validate manager reference integrity and onboarding flow.'}
audit = pd.DataFrame([{
'issue_type':k,'affected_rows':int(v.sum()),'severity':sev[k],'recommended_next_action':rec[k]} for k,v in rules.items()])
audit


,issue_type,affected_rows,severity,recommended_next_action
0,required_employee_id,1,High,Escalate and resolve missing IDs before joins.
1,age_between_0_120,1,High,Verify with HR master records and correct sour...
2,salary_non_negative,2,High,Investigate extract mapping and payroll source...
3,hire_date_valid,1,Medium,Enforce standard date format at entry point.
4,duplicate_employee_id,2,High,Investigate key-generation process and dedup p...
5,known_department,2,Medium,Reconcile against department reference table.
6,contract_standardized,1,Low,Align labels to canonical categories.
7,manager_reference_exists,2,Medium,Validate manager reference integrity and onboa...


## Step 4 - Prioritized summary


In [7]:
rank={'High':3,'Medium':2,'Low':1}
audit['rank']=audit['severity'].map(rank)
summary=audit.sort_values(['rank','affected_rows'], ascending=[False,False]).drop(columns=['rank'])
summary


,issue_type,affected_rows,severity,recommended_next_action
2,salary_non_negative,2,High,Investigate extract mapping and payroll source...
4,duplicate_employee_id,2,High,Investigate key-generation process and dedup p...
0,required_employee_id,1,High,Escalate and resolve missing IDs before joins.
1,age_between_0_120,1,High,Verify with HR master records and correct sour...
5,known_department,2,Medium,Reconcile against department reference table.
7,manager_reference_exists,2,Medium,Validate manager reference integrity and onboa...
3,hire_date_valid,1,Medium,Enforce standard date format at entry point.
6,contract_standardized,1,Low,Align labels to canonical categories.


In [8]:
records_with_any = pd.concat(rules.values(), axis=1).any(axis=1).sum()
print(f'Total records: {len(df)}')
print(f'Records with >=1 issue: {records_with_any} ({records_with_any/len(df):.1%})')


Total records: 10
Records with >=1 issue: 9 (90.0%)


## Function spotlight - reusable rule-audit pattern

In real projects, we keep rule logic in functions so audits are reproducible.


In [9]:
def evaluate_rule(rule_name, mask, severity, recommendation):
    """Create one audit row from a validation mask.

    Parameters
    ----------
    rule_name : str
        Unique name of the rule.
    mask : pd.Series (bool)
        Boolean vector where True means rule violation.
    severity : str
        Suggested priority level (High, Medium, Low).
    recommendation : str
        Next action for data owner or engineering team.

    Returns
    -------
    dict
        A dictionary that can be turned into one row of an audit report.
    """
    return {
        'issue_type': rule_name,
        'affected_rows': int(mask.sum()),
        'severity': severity,
        'recommended_next_action': recommendation,
    }


def build_audit_report(rule_dict, severity_map, recommendation_map):
    """Build a full audit report table from rule definitions.

    Parameters
    ----------
    rule_dict : dict[str, pd.Series]
        Mapping from rule name to violation mask.
    severity_map : dict[str, str]
        Severity per rule.
    recommendation_map : dict[str, str]
        Recommendation text per rule.

    Returns
    -------
    pd.DataFrame
        Audit report with one row per rule.
    """
    rows = [
        evaluate_rule(name, mask, severity_map[name], recommendation_map[name])
        for name, mask in rule_dict.items()
    ]
    return pd.DataFrame(rows)


audit_v2 = build_audit_report(rules, sev, rec)
audit_v2


,issue_type,affected_rows,severity,recommended_next_action
0,required_employee_id,1,High,Escalate and resolve missing IDs before joins.
1,age_between_0_120,1,High,Verify with HR master records and correct sour...
2,salary_non_negative,2,High,Investigate extract mapping and payroll source...
3,hire_date_valid,1,Medium,Enforce standard date format at entry point.
4,duplicate_employee_id,2,High,Investigate key-generation process and dedup p...
5,known_department,2,Medium,Reconcile against department reference table.
6,contract_standardized,1,Low,Align labels to canonical categories.
7,manager_reference_exists,2,Medium,Validate manager reference integrity and onboa...


## Governance reflection

1. Which high-severity issue should be fixed first?
2. Which rules should run automatically in ingestion?
3. How does this audit support accountability?


## Recap

Validation rules create measurable controls, while audits translate technical findings into governance-ready decisions.
